In [ ]:
%%sql -r dataframe_1
CREATE DATABASE IF NOT EXISTS SALES_ML_DB;

CREATE SCHEMA IF NOT EXISTS SALES_ML_DB.SALES_PREDICTION;

CREATE WAREHOUSE IF NOT EXISTS SALES_ML_WH
WITH WAREHOUSE_SIZE = 'XSMALL'
AUTO_SUSPEND = 60
AUTO_RESUME = TRUE;

USE DATABASE SALES_ML_DB;
USE SCHEMA SALES_PREDICTION;
USE WAREHOUSE SALES_ML_WH;

In [ ]:
%%sql -r dataframe_2
SELECT
    CURRENT_DATABASE() AS DATABASE_NAME,
    CURRENT_SCHEMA() AS SCHEMA_NAME,
    CURRENT_WAREHOUSE() AS WAREHOUSE_NAME;

In [ ]:

import os
import pandas as pd

# Check uploaded files
print("Files available in notebook:")

for file in os.listdir():
    print(file)


In [ ]:

# Load the four datasets

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
store_df = pd.read_csv("store.csv")
submission_df = pd.read_csv("sample_submission.csv")

# Display dataset dimensions

print("Training Data:", train_df.shape)
print("Testing Data:", test_df.shape)
print("Store Data:", store_df.shape)
print("Submission Data:", submission_df.shape)


In [ ]:

# Display training dataset columns
print("TRAIN DATA COLUMNS:")
print(train_df.columns.tolist())

# Display first 5 records
print("\nFIRST 5 TRAINING RECORDS:")
display(train_df.head())

# Display store dataset
print("\nSTORE DATA:")
display(store_df.head())


In [ ]:

# Check missing values in all datasets

print("TRAIN MISSING VALUES")
display(train_df.isnull().sum())

print("TEST MISSING VALUES")
display(test_df.isnull().sum())

print("STORE MISSING VALUES")
display(store_df.isnull().sum())


In [ ]:

# Create copies to preserve original datasets
train_clean = train_df.copy()
test_clean = test_df.copy()
store_clean = store_df.copy()

# 1. Handle missing values in test dataset
test_clean["Open"] = test_clean["Open"].fillna(1)

# 2. Handle missing CompetitionDistance
store_clean["CompetitionDistance"] = (
    store_clean["CompetitionDistance"].fillna(
        store_clean["CompetitionDistance"].median()
    )
)

# 3. Handle missing competition opening information
store_clean["CompetitionOpenSinceMonth"] = (
    store_clean["CompetitionOpenSinceMonth"].fillna(0)
)

store_clean["CompetitionOpenSinceYear"] = (
    store_clean["CompetitionOpenSinceYear"].fillna(0)
)

# 4. Handle missing Promo2 information
store_clean["Promo2SinceWeek"] = (
    store_clean["Promo2SinceWeek"].fillna(0)
)

store_clean["Promo2SinceYear"] = (
    store_clean["Promo2SinceYear"].fillna(0)
)

store_clean["PromoInterval"] = (
    store_clean["PromoInterval"].fillna("None")
)

print("Missing value treatment completed!")


In [ ]:

print("TRAIN DATA:")
print(train_clean.isnull().sum())

print("\nTEST DATA:")
print(test_clean.isnull().sum())

print("\nSTORE DATA:")
print(store_clean.isnull().sum())


In [ ]:

# Merge training and store datasets
train_merged = pd.merge(
    train_clean,
    store_clean,
    on="Store",
    how="left",
    validate="many_to_one"
)

# Merge testing and store datasets
test_merged = pd.merge(
    test_clean,
    store_clean,
    on="Store",
    how="left",
    validate="many_to_one"
)

# Convert Date column into datetime format
train_merged["Date"] = pd.to_datetime(train_merged["Date"])
test_merged["Date"] = pd.to_datetime(test_merged["Date"])

print("Training dataset after merging:", train_merged.shape)
print("Testing dataset after merging:", test_merged.shape)

display(train_merged.head())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic statistical summary
print("Sales Statistics:")
display(train_merged["Sales"].describe())

# Sales distribution
plt.figure(figsize=(8, 5))
sns.histplot(train_merged["Sales"], bins=50, kde=True)

plt.title("Distribution of Store Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

In [ ]:
promo_sales = train_merged.groupby("Promo")["Sales"].mean()

print(promo_sales)

promo_sales.plot(
    kind="bar",
    figsize=(6, 4),
    title="Average Sales: With and Without Promotion"
)

plt.xlabel("Promotion (0 = No, 1 = Yes)")
plt.ylabel("Average Sales")
plt.show()

In [ ]:
# Extract month and year
train_merged["Year"] = train_merged["Date"].dt.year
train_merged["Month"] = train_merged["Date"].dt.month

# Calculate average monthly sales
monthly_sales = train_merged.groupby("Month")["Sales"].mean()

print("Average Sales by Month:")
print(monthly_sales)

# Visualize monthly sales
plt.figure(figsize=(9, 5))
monthly_sales.plot(kind="line", marker="o")

plt.title("Average Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.xticks(range(1, 13))
plt.grid(True)
plt.show()

In [ ]:
# Calculate average sales by day of the week
weekday_sales = train_merged.groupby("DayOfWeek")["Sales"].mean()

print("Average Sales by Day of Week:")
print(weekday_sales)

# Visualize the results
plt.figure(figsize=(8, 5))

weekday_sales.plot(kind="bar")

plt.title("Average Sales by Day of the Week")
plt.xlabel("Day of Week (1 = Monday, 7 = Sunday)")
plt.ylabel("Average Sales")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Calculate average sales by store type
storetype_sales = train_merged.groupby("StoreType")["Sales"].mean()

print("Average Sales by Store Type:")
print(storetype_sales)

# Visualize
plt.figure(figsize=(7, 5))

storetype_sales.plot(kind="bar")

plt.title("Average Sales by Store Type")
plt.xlabel("Store Type")
plt.ylabel("Average Sales")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Calculate average sales by assortment
assortment_sales = train_merged.groupby("Assortment")["Sales"].mean()

print("Average Sales by Assortment:")
print(assortment_sales)

# Visualize
plt.figure(figsize=(7, 5))

assortment_sales.plot(kind="bar")

plt.title("Average Sales by Assortment")
plt.xlabel("Assortment Type")
plt.ylabel("Average Sales")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Calculate average sales based on store opening status
open_sales = train_merged.groupby("Open")["Sales"].mean()

print("Average Sales by Store Opening Status:")
print(open_sales)

# Visualize
plt.figure(figsize=(6, 4))

open_sales.plot(kind="bar")

plt.title("Average Sales by Store Opening Status")
plt.xlabel("Open (0 = Closed, 1 = Open)")
plt.ylabel("Average Sales")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Select numerical columns for correlation analysis
numeric_data = train_merged[
    [
        "Sales",
        "Store",
        "DayOfWeek",
        "Open",
        "Promo",
        "SchoolHoliday",
        "CompetitionDistance",
        "Promo2"
    ]
]

# Calculate correlation
correlation = numeric_data.corr()

print("Correlation with Sales:")
print(correlation["Sales"].sort_values(ascending=False))

# Visualize correlation matrix
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Create date-related features

for df in [train_merged, test_merged]:

    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["IsWeekend"] = df["DayOfWeek"].isin([6, 7]).astype(int)

print("Date feature engineering completed!")

# Display the new features
display(
    train_merged[
        ["Date", "Year", "Month", "Day", "DayOfWeek",
         "WeekOfYear", "IsWeekend"]
    ].head()
)

In [ ]:
# Define target variable
target = "Sales"

# Select features for prediction
features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "StateHoliday",
    "SchoolHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "IsWeekend"
]

# Create X and y
X = train_merged[features].copy()
y = train_merged[target].copy()

# Create test input
X_test = test_merged[features].copy()

print("Training Features Shape:", X.shape)
print("Target Shape:", y.shape)
print("Testing Features Shape:", X_test.shape)

print("\nSelected Features:")
print(features)

In [ ]:
# Identify categorical columns
categorical_features = [
    "StateHoliday",
    "StoreType",
    "Assortment",
    "PromoInterval",
    "Store"
]

# Identify numerical columns
numerical_features = [
    col for col in features
    if col not in categorical_features
]

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Data Types:")
print(X[categorical_features].dtypes)

print("\nUnique Categorical Values:")
for col in categorical_features:
    print(f"{col}: {X[col].nunique()} unique values")

In [ ]:
# Display unique values in categorical columns

for col in categorical_features:
    print(f"\n{col}")
    print(X[col].value_counts(dropna=False).head(15))

In [ ]:
# 1. Normalize categorical features

for col in categorical_features:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print("Categorical values normalized!")

print("\nStateHoliday categories:")
print(X["StateHoliday"].value_counts())


# 2. Create chronological training and validation split

validation_start = (
    train_merged["Date"].max() - pd.Timedelta(days=42)
)

train_mask = train_merged["Date"] < validation_start
val_mask = train_merged["Date"] >= validation_start

X_train = X.loc[train_mask].copy()
X_val = X.loc[val_mask].copy()

y_train = y.loc[train_mask].copy()
y_val = y.loc[val_mask].copy()


# 3. Display dataset information

print("\nTraining Dataset:", X_train.shape)
print("Validation Dataset:", X_val.shape)

print("\nTraining Date Range:")
print(train_merged.loc[train_mask, "Date"].min(),
      "to",
      train_merged.loc[train_mask, "Date"].max())

print("\nValidation Date Range:")
print(train_merged.loc[val_mask, "Date"].min(),
      "to",
      train_merged.loc[val_mask, "Date"].max())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

In [ ]:
from sklearn.linear_model import LinearRegression

# Create the Linear Regression pipeline
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train the model
print("Training Linear Regression model...")

linear_model.fit(X_train, y_train)

print("Linear Regression model trained successfully!")

In [ ]:
# Predict sales on validation data
y_pred = linear_model.predict(X_val)

print("First 10 Actual Sales:")
print(y_val.head(10).values)

print("\nFirst 10 Predicted Sales:")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

# Calculate evaluation metrics
mae = mean_absolute_error(y_val, y_pred)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))

r2 = r2_score(y_val, y_pred)

# Display results
print("LINEAR REGRESSION MODEL EVALUATION")
print("----------------------------------")
print("Mean Absolute Error (MAE):", round(mae, 2))
print("Root Mean Squared Error (RMSE):", round(rmse, 2))
print("R2 Score:", round(r2, 4))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create Random Forest model
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features=0.7,
        n_jobs=-1,
        random_state=42
    ))
])

# Train model
print("Training Random Forest model...")

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
# Predict validation sales
rf_pred = rf_model.predict(X_val)

print("First 10 Actual Sales:")
print(y_val.head(10).values)

print("\nFirst 10 Random Forest Predictions:")
print(rf_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

# Calculate evaluation metrics
rf_mae = mean_absolute_error(y_val, rf_pred)

rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))

rf_r2 = r2_score(y_val, rf_pred)

# Display results
print("RANDOM FOREST MODEL EVALUATION")
print("--------------------------------")
print("Mean Absolute Error (MAE):", round(rf_mae, 2))
print("Root Mean Squared Error (RMSE):", round(rf_rmse, 2))
print("R2 Score:", round(rf_r2, 4))

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

et_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", ExtraTreesRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features=0.7,
        n_jobs=-1,
        random_state=42
    ))
])

print("Training Extra Trees model...")

et_model.fit(X_train, y_train)

print("Extra Trees model trained successfully!")

In [ ]:
et_pred = et_model.predict(X_val)

In [ ]:
et_mae = mean_absolute_error(y_val, et_pred)
et_rmse = np.sqrt(mean_squared_error(y_val, et_pred))
et_r2 = r2_score(y_val, et_pred)

print("EXTRA TREES MODEL EVALUATION")
print("--------------------------------")
print("MAE:", round(et_mae, 2))
print("RMSE:", round(et_rmse, 2))
print("R2 Score:", round(et_r2, 4))

In [ ]:
# Create the final Linear Regression model

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

print("Training final model...")

final_model.fit(X, y)

print("Final model trained successfully!")

In [ ]:
# Generate sales predictions
final_predictions = final_model.predict(X_test)

# Prevent negative sales predictions
final_predictions = np.maximum(final_predictions, 0)

# Set sales to zero for closed stores
final_predictions[test_merged["Open"].values == 0] = 0

print("Sales predictions generated successfully!")

print("\nFirst 10 predicted sales:")
print(final_predictions[:10])

print("\nTotal predictions:", len(final_predictions))

In [ ]:
# Create final submission dataframe
submission = submission_df.copy()

# Insert predicted sales
submission["Sales"] = final_predictions

# Save as CSV
submission.to_csv("sales_predictions.csv", index=False)

print("Submission file created successfully!")

print("\nFirst 10 rows:")
print(submission.head(10))

print("\nFile shape:", submission.shape)

print("\nColumn names:")
print(submission.columns.tolist())

In [ ]:
import os

file_path = "sales_predictions.csv"

if os.path.exists(file_path):
    print("File exists successfully!")
    print("File size:", round(os.path.getsize(file_path) / 1024, 2), "KB")
else:
    print("File not found!")

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake session connected successfully!")

In [ ]:
# Write predictions into Snowflake table

session.write_pandas(
    submission,
    table_name="SALES_PREDICTIONS",
    database="SALES_ML_DB",
    schema="SALES_PREDICTION",
    auto_create_table=True,
    overwrite=True
)

print("Predictions saved to Snowflake successfully!")

In [ ]:
SELECT *
FROM SALES_ML_DB.SALES_PREDICTION.SALES_PREDICTIONS
LIMIT 10;

In [ ]:
DESC TABLE SALES_ML_DB.SALES_PREDICTION.SALES_PREDICTIONS;

In [ ]:
SELECT
    COUNT(*) AS TOTAL_RECORDS,
    MIN("Sales") AS MINIMUM_SALES,
    MAX("Sales") AS MAXIMUM_SALES,
    AVG("Sales") AS AVERAGE_SALES
FROM SALES_ML_DB.SALES_PREDICTION.SALES_PREDICTIONS;

In [ ]:
import joblib
import os

# Save the trained model
joblib.dump(final_model, "sales_model.joblib")

# Create a Snowflake stage for the model
session.sql("""
    CREATE STAGE IF NOT EXISTS
    SALES_ML_DB.SALES_PREDICTION.MODEL_STAGE
""").collect()

# Upload the model to Snowflake
model_path = os.path.abspath("sales_model.joblib")

session.file.put(
    model_path,
    "@SALES_ML_DB.SALES_PREDICTION.MODEL_STAGE",
    auto_compress=False,
    overwrite=True
)

print("Trained model saved to Snowflake!")

In [ ]:
session.write_pandas(
    store_clean,
    table_name="STORE_DETAILS",
    database="SALES_ML_DB",
    schema="SALES_PREDICTION",
    auto_create_table=True,
    overwrite=True
)

print("Store details saved to Snowflake!")

In [ ]:
import sklearn
import joblib
import pandas as pd

print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)
print("Pandas:", pd.__version__)

In [ ]:
LIST @SALES_ML_DB.SALES_PREDICTION.MODEL_STAGE;

In [ ]:
SELECT *
FROM SALES_ML_DB.SALES_PREDICTION.STORE_DETAILS
LIMIT 5;

In [ ]:
SELECT COUNT(*)
FROM SALES_ML_DB.SALES_PREDICTION.SALES_PREDICTIONS;

In [ ]:
SELECT
    CURRENT_USER() AS USER_NAME,
    CURRENT_ROLE() AS CURRENT_ROLE,
    CURRENT_DATABASE() AS DATABASE_NAME,
    CURRENT_WAREHOUSE() AS WAREHOUSE_NAME;

In [ ]:
SHOW DATABASE ROLES IN DATABASE SNOWFLAKE;

In [ ]:
SELECT CURRENT_ROLE();

In [ ]:
SHOW GRANTS TO ROLE ACCOUNTADMIN;

In [ ]:
SHOW GRANTS OF DATABASE ROLE SNOWFLAKE.PYPI_REPOSITORY_USER;

In [ ]:
SHOW DATABASE ROLES IN DATABASE SNOWFLAKE;

In [ ]:
%%sql -r dataframe_16

CREATE OR REPLACE API INTEGRATION GITHUB_API_INTEGRATION
    API_PROVIDER = git_https_api
    API_ALLOWED_PREFIXES = ('https://github.com/Myth-Arise')
    API_USER_AUTHENTICATION = (
        TYPE = SNOWFLAKE_GITHUB_APP
    )
    ENABLED = TRUE;
